# Task 156: kilosort_spike — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Spike sorting using Kilosort template matching

**Paper**: Kilosort4: accurate spike sorting by template matching
**Repository**: https://github.com/MouseLand/Kilosort

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: N/A (spike sorting)
- **SSIM**: N/A
- **Evaluation**: Spike sorting (Accuracy=0.9904, F1=0.9765, Detection Rate=0.9541)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
kilosort_spike - Spike Sorting Inverse Problem
===============================================
Task: Detect and classify neural spikes from multi-channel electrophysiology
Repo: https://github.com/MouseLand/Kilosort
"""
import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`bandpass_filter`, `detect_spikes`, `extract_waveforms`, `sort_spikes`, `match_clusters`

In [ ]:
def bandpass_filter(data, low=300, high=6000, fs=30000, order=3):
    """Apply bandpass filter."""
    nyq = fs / 2
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data, axis=0)

def detect_spikes(filtered, fs=30000, threshold_factor=4.0, n_template_samples=61):
    """Detect spikes by threshold crossing on peak channel."""
    half = n_template_samples // 2
    # Use channel with highest variance
    energy = np.std(filtered, axis=0)
    
    # Multi-channel detection: sum of squares
    detection_signal = np.sum(filtered**2, axis=1)
    threshold = np.median(detection_signal) + threshold_factor * np.std(detection_signal)
    
    # Find peaks above threshold
    above = detection_signal > threshold
    crossings = np.where(np.diff(above.astype(int)) == 1)[0]
    
    # Refine: find local minima (negative peak) near each crossing
    spike_times = []
    refractory = int(0.001 * fs)  # 1ms
    last_spike = -refractory
    
    for c in crossings:
        window_start = max(0, c - half)
        window_end = min(len(detection_signal), c + half)
        peak_idx = window_start + np.argmax(detection_signal[window_start:window_end])
        if peak_idx - last_spike > refractory and peak_idx > half and peak_idx < len(filtered) - half:
            spike_times.append(peak_idx)
            last_spike = peak_idx
    
    return np.array(spike_times)

def extract_waveforms(filtered, spike_times, n_template_samples=61):
    """Extract waveform snippets around detected spikes."""
    half = n_template_samples // 2
    n_channels = filtered.shape[1]
    waveforms = []
    valid_times = []
    for st in spike_times:
        start = st - half
        end = st + half + 1
        if start >= 0 and end <= len(filtered):
            wf = filtered[start:end, :].flatten()
            waveforms.append(wf)
            valid_times.append(st)
    return np.array(waveforms), np.array(valid_times)

def sort_spikes(waveforms, n_clusters, n_pca_components=5):
    """Sort spikes using PCA + KMeans."""
    pca = PCA(n_components=min(n_pca_components, waveforms.shape[1], waveforms.shape[0]))
    features = pca.fit_transform(waveforms)
    
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = kmeans.fit_predict(features)
    
    return labels, features, pca

def match_clusters(pred_labels, pred_times, gt_labels, gt_times, n_neurons, tolerance=15):
    """Match predicted clusters to GT neurons using Hungarian algorithm."""
    n_clusters = len(np.unique(pred_labels))
    cost_matrix = np.zeros((n_clusters, n_neurons))
    
    for i, pt in enumerate(pred_times):
        dists = np.abs(gt_times.astype(float) - float(pt))
        if len(dists) > 0 and np.min(dists) < tolerance:
            gt_idx = np.argmin(dists)
            gt_label = gt_labels[gt_idx]
            pred_label = pred_labels[i]
            cost_matrix[pred_label, gt_label] += 1
    
    row_ind, col_ind = linear_sum_assignment(-cost_matrix)
    mapping = {}
    for r, c in zip(row_ind, col_ind):
        mapping[r] = c
    
    return mapping

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_spike_template`, `synthesize_recording`

In [ ]:
def generate_spike_template(n_samples=61, template_type=0):
    """Generate realistic biphasic spike waveform templates."""
    t = np.linspace(-1, 2, n_samples)
    if template_type == 0:
        template = -np.exp(-t**2/0.1) + 0.3*np.exp(-(t-0.5)**2/0.2)
    elif template_type == 1:
        template = -0.8*np.exp(-t**2/0.08) + 0.5*np.exp(-(t-0.4)**2/0.15)
    elif template_type == 2:
        template = -1.2*np.exp(-t**2/0.12) + 0.2*np.exp(-(t-0.6)**2/0.25)
    else:
        template = -0.6*np.exp(-t**2/0.06) + 0.4*np.exp(-(t-0.3)**2/0.1)
    return template / np.abs(template).max()

def synthesize_recording(n_neurons=4, n_channels=8, fs=30000, duration=10.0, 
                         spike_rate=3.0, noise_level=0.3, seed=42):
    """Synthesize multi-channel recording with known spike times."""
    np.random.seed(seed)
    n_samples = int(fs * duration)
    n_template_samples = 61  # ~2ms at 30kHz
    half_template = n_template_samples // 2
    
    recording = np.random.randn(n_samples, n_channels) * noise_level
    
    gt_spike_times = []
    gt_spike_labels = []
    templates = []
    spatial_profiles = []
    
    for neuron_id in range(n_neurons):
        # Template
        template = generate_spike_template(n_template_samples, neuron_id)
        amplitude = 1.0 + 0.5 * neuron_id
        template *= amplitude
        templates.append(template)
        
        # Spatial profile (different for each neuron)
        spatial = np.random.rand(n_channels)
        peak_ch = neuron_id % n_channels
        for ch in range(n_channels):
            dist = min(abs(ch - peak_ch), n_channels - abs(ch - peak_ch))
            spatial[ch] = np.exp(-dist**2 / 2.0)
        spatial /= spatial.max()
        spatial_profiles.append(spatial)
        
        # Spike times (Poisson process with refractory period)
        n_spikes_expected = int(spike_rate * duration)
        isi = np.random.exponential(fs / spike_rate, n_spikes_expected * 2)
        isi = np.maximum(isi, int(0.002 * fs))  # 2ms refractory
        spike_times = np.cumsum(isi).astype(int)
        spike_times = spike_times[spike_times < n_samples - n_template_samples]
        spike_times = spike_times[:n_spikes_expected]
        
        for st in spike_times:
            start = st - half_template
            end = st + half_template + 1
            if start >= 0 and end <= n_samples:
                for ch in range(n_channels):
                    recording[start:end, ch] += template * spatial[ch]
                gt_spike_times.append(st)
                gt_spike_labels.append(neuron_id)
    
    sort_idx = np.argsort(gt_spike_times)
    gt_spike_times = np.array(gt_spike_times)[sort_idx]
    gt_spike_labels = np.array(gt_spike_labels)[sort_idx]
    
    return recording, gt_spike_times, gt_spike_labels, templates, spatial_profiles

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_sorting`, `visualize_results`

In [ ]:
def evaluate_sorting(pred_labels, pred_times, gt_labels, gt_times, mapping, tolerance=15):
    """Evaluate spike sorting accuracy."""
    correct = 0
    detected = 0
    
    for i, pt in enumerate(pred_times):
        dists = np.abs(gt_times.astype(float) - float(pt))
        if len(dists) > 0 and np.min(dists) < tolerance:
            gt_idx = np.argmin(dists)
            detected += 1
            pred_neuron = mapping.get(pred_labels[i], -1)
            if pred_neuron == gt_labels[gt_idx]:
                correct += 1
    
    accuracy = correct / max(detected, 1)
    detection_rate = detected / max(len(gt_times), 1)
    precision = detected / max(len(pred_times), 1)
    recall = detected / max(len(gt_times), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-10)
    
    return {
        'accuracy': float(accuracy),
        'detection_rate': float(detection_rate),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'n_gt_spikes': int(len(gt_times)),
        'n_detected_spikes': int(len(pred_times)),
        'n_correct': int(correct)
    }

def visualize_results(recording, filtered, gt_times, gt_labels, pred_times, pred_labels, 
                      features, mapping, metrics, fs=30000, save_path='results/reconstruction_result.png'):
    """Generate visualization."""
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    
    # Panel 1: Raw recording snippet (first 1 second, 3 channels)
    ax = axes[0, 0]
    t = np.arange(min(fs, len(recording))) / fs
    n_show = min(3, recording.shape[1])
    for ch in range(n_show):
        ax.plot(t, recording[:len(t), ch] + ch * 3, 'k', linewidth=0.3, alpha=0.7)
    ax.set_title('Raw Multi-Channel Recording (1s)')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Channel')
    
    # Panel 2: PCA feature space with cluster colors
    ax = axes[0, 1]
    if features.shape[1] >= 2:
        colors = plt.cm.tab10(np.linspace(0, 1, len(np.unique(pred_labels))))
        for label in np.unique(pred_labels):
            mask = pred_labels == label
            neuron_id = mapping.get(label, label)
            ax.scatter(features[mask, 0], features[mask, 1], 
                      c=[colors[label]], alpha=0.5, s=10, label=f'Cluster {label}→Neuron {neuron_id}')
    ax.set_title('PCA Feature Space')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(fontsize=8)
    
    # Panel 3: Raster plot — GT
    ax = axes[1, 0]
    n_neurons = len(np.unique(gt_labels))
    colors_gt = plt.cm.tab10(np.linspace(0, 1, n_neurons))
    for neuron_id in range(n_neurons):
        mask = gt_labels == neuron_id
        times = gt_times[mask] / fs
        ax.scatter(times, np.full_like(times, neuron_id), c=[colors_gt[neuron_id]], s=2, marker='|')
    ax.set_title('Ground Truth Spike Raster')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Neuron ID')
    ax.set_yticks(range(n_neurons))
    
    # Panel 4: Raster plot — Sorted
    ax = axes[1, 1]
    for label in np.unique(pred_labels):
        mask = pred_labels == label
        times = pred_times[mask] / fs
        neuron_id = mapping.get(label, label)
        ax.scatter(times, np.full_like(times, neuron_id), c=[colors_gt[neuron_id % len(colors_gt)]], s=2, marker='|')
    ax.set_title('Sorted Spike Raster')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Neuron ID')
    ax.set_yticks(range(n_neurons))
    
    fig.suptitle(f"Spike Sorting | Accuracy={metrics['accuracy']:.3f} | F1={metrics['f1']:.3f} | "
                 f"Detected={metrics['n_detected_spikes']}/{metrics['n_gt_spikes']}", fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved → {save_path}")

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("  kilosort_spike — Spike Sorting Pipeline")
print("=" * 60)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
N_NEURONS = 4
N_CHANNELS = 8
FS = 30000
DURATION = 10.0

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Generate synthetic data
print("[DATA] Synthesizing multi-channel recording...")
recording, gt_times, gt_labels, templates, spatial_profiles = synthesize_recording(
    n_neurons=N_NEURONS, n_channels=N_CHANNELS, fs=FS, duration=DURATION)
print(f"[DATA] Recording: {recording.shape}, GT spikes: {len(gt_times)}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Bandpass filter
print("[FILT] Bandpass filtering (300-6000 Hz)...")
filtered = bandpass_filter(recording, fs=FS)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Detect spikes
print("[DET] Detecting spikes...")
pred_times = detect_spikes(filtered, fs=FS)
print(f"[DET] Detected {len(pred_times)} spikes (GT: {len(gt_times)})")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Extract waveforms
waveforms, valid_times = extract_waveforms(filtered, pred_times)
print(f"[EXT] Extracted {len(waveforms)} waveforms")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Sort spikes
print("[SORT] Clustering spikes...")
pred_labels, features, pca = sort_spikes(waveforms, n_clusters=N_NEURONS)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Match clusters to GT
mapping = match_clusters(pred_labels, valid_times, gt_labels, gt_times, N_NEURONS)
print(f"[MATCH] Cluster mapping: {mapping}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Evaluate
metrics = evaluate_sorting(pred_labels, valid_times, gt_labels, gt_times, mapping)
print(f"[EVAL] Accuracy: {metrics['accuracy']:.4f}")
print(f"[EVAL] F1: {metrics['f1']:.4f}")
print(f"[EVAL] Detection rate: {metrics['detection_rate']:.4f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Save metrics
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print(f"[SAVE] Metrics → results/metrics.json")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Save arrays
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), gt_labels)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), pred_labels)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Visualize
vis_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
visualize_results(recording, filtered, gt_times, gt_labels, valid_times, 
                 pred_labels, features, mapping, metrics, fs=FS, save_path=vis_path)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("  DONE")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **kilosort_spike**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=N/A (spike sorting), SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Kilosort4: accurate spike sorting by template matching
- Repository: https://github.com/MouseLand/Kilosort
